Groundwater | Transport Modeling Track

# Step 3: From Perceptual to Conceptual — MODFLOW 6 for Solute Transport

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → **3-MODFLOW** → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Step 2, you built a perceptual model of **conservative-tracer transport** through the Limmat Valley aquifer — a tracer moving between the injection and abstraction wells of a **groundwater heat-exchange (GWHE) doublet** (paired wells used for heating/cooling, studied here as a solute-transport problem). You estimated a domain-average seepage velocity of ~2.3 m/d, set an effective porosity of $n_e = 0.20$ and a longitudinal dispersivity $\alpha_L = 10$ m, and framed the operational question: **does what we inject short-circuit back to the abstraction well, and how fast?** Now we translate these insights into the language of MODFLOW 6: the transport equation, numerical stability criteria, and the GWT solute-transport packages.

## Timing and Learning Objectives

| | |
|---|---|
| **Time** | ~25–35 min core + 10 min optional sections |
| **Prerequisites** | Flow track complete, Transport Steps 1–2 |

**Learning objectives:**

By the end of this notebook you will be able to:

1. Write the **advection-dispersion equation (ADE)** for a dissolved solute and explain each term
2. Evaluate **grid Peclet and Courant numbers** for transport stability
3. Identify **MODFLOW 6 GWT packages** and their roles
4. Map **perceptual transport features** to specific GWT package inputs

> 📌 **You MODIFY a provided model.** Throughout the transport track you start from an instructor-supplied GWT model and change its transport parameters and source terms — you do not build the grid or refine it yourself. Grid design and refinement are revisited in Step 4.

In [ ]:
# Setup
import sys

sys.path.append('../../_SUPPORT/src')
sys.path.append('../../_SUPPORT/src/scripts/scripts_exercises')

from shared_functions import check_task_with_solution, create_multiple_choice

---

## Introduction

In the flow track (Step 3), you learned how MODFLOW 6 discretises the **flow equation** — control volumes, hydraulic conductivity averaging, grid types, and boundary condition packages. All of that carries over unchanged to a transport simulation: **the same grid, the same flow solution, the same boundary conditions drive transport.**

What's new for transport?

| Already covered (flow NB3) | New in this notebook |
|---|---|
| Governing flow equation | Advection-dispersion equation (ADE) |
| CVFD discretisation, K-averaging | Numerical stability (Peclet, Courant) |
| Grid types (DIS, DISV, DISU) | Transport-specific packages (GWT) |
| GWF packages (RIV, RCH, WEL, …) | How GWF fluxes carry solute concentration (SSM) |

> 💡 **Why MODFLOW 6 for transport?** The Groundwater Transport (GWT) model is fully integrated into MODFLOW 6. It shares the same grid as the Groundwater Flow (GWF) model, runs in the same simulation, and is fully scriptable via FloPy — no separate transport code needed. (A sister model, GWE, does the same for heat; see the optional box at the end of this notebook.)

---

## 1 — The Advection-Dispersion Equation

### Governing equation

The **advection-dispersion equation (ADE)** describes how a dissolved solute is transported through a porous medium:

$$\frac{\partial (n C)}{\partial t} = \nabla \cdot (n \mathbf{D} \nabla C) - \nabla \cdot (n \mathbf{v} C) + q_s$$

| Term | Name | Physical meaning |
|---|---|---|
| $\partial(nC)/\partial t$ | **Storage** | Change in dissolved mass stored per unit volume |
| $\nabla \cdot (n \mathbf{D} \nabla C)$ | **Dispersion** | Spreading due to velocity variations + molecular diffusion |
| $\nabla \cdot (n \mathbf{v} C)$ | **Advection** | Bulk movement with the flowing groundwater |
| $q_s$ | **Sources / sinks** | Mass added or removed (wells, rivers, reactions) |

where $n$ is porosity, $C$ is solute concentration, $\mathbf{v}$ is the seepage velocity vector, and $\mathbf{D}$ is the hydrodynamic dispersion tensor. For a **conservative tracer** there are no sorption or decay terms — this is the core path for this track. (Optional reactive solutes, where storage also includes a sorbed mass term, are introduced in Step 2's optional box and handled by the **MST** package.)

### Flow vs. transport: a comparison

| | Flow equation | Transport equation |
|---|---|---|
| **State variable** | Hydraulic head $h$ | Concentration $C$ |
| **What drives movement** | Hydraulic gradient $\nabla h$ | Groundwater velocity $\mathbf{v}$ (advection) + gradient $\nabla C$ (dispersion) |
| **What spreads it** | Hydraulic conductivity $K$ | Dispersion tensor $\mathbf{D}$ (= $\alpha_L v$ + molecular diffusion $D_m^*$) |
| **Storage** | Specific storage $S_s$ | Porosity $n$ (+ sorption, for reactive solutes) |
| **Coupling** | Independent | **One-directional:** flow → transport (velocity field) |

> 💡 **Transport is coupled but one-directional.** The flow model produces a velocity field that drives advection. For a dilute, conservative tracer the solute does not change the water density and therefore does not feed back on the flow field. (Density-dependent flow — brines, strong thermal gradients — is a separate topic not covered here.)

### A note on the 1D analytical benchmark

The Week 7–8 theory lectures solve the **one-dimensional ADE** for uniform flow and a step input. The classic solution is **Ogata–Banks**, in its full two-term first-type (constant-concentration inlet) form:

$$\frac{C}{C_0} = \tfrac{1}{2}\,\mathrm{erfc}\!\left[\frac{x - vt}{2\sqrt{D_L t}}\right] + \tfrac{1}{2}\,\exp\!\left(\frac{vx}{D_L}\right)\mathrm{erfc}\!\left[\frac{x + vt}{2\sqrt{D_L t}}\right]$$

The common **single-term** simplification (dropping the second term) is only accurate at **large Peclet number**; keep the full two-term form for a quantitative check.

> ⚠️ **Why we will not overlay this on the doublet.** Ogata–Banks assumes **1D uniform flow**. The injection–abstraction doublet produces a **convergent, non-uniform** flow field, so a clean Ogata–Banks match is expected only for a dedicated **1D uniform-flow verification toy** — built in Step 8 — and *not* for the doublet breakthrough. For the doublet, the swept-pore-volume travel time ($V_{pore}/Q$) is a single **mean / characteristic residence time**, not the first-arrival time: breakthrough begins earlier (along the fastest streamline) and tails out behind it. Treat the doublet's departure from Ogata–Banks as a **limitation of the analytical idealisation, not a model failure**. *(This framing assumes the Week 7–8 lectures teach the 1D ADE / Ogata–Banks solution — your instructor will confirm.)*

In [ ]:
# Checkpoint 1 — ADE Comprehension
create_multiple_choice('task_t03_checkpoint_1')

---

## 2 — Numerical Stability: Peclet and Courant Numbers

Transport is numerically **harder** than flow. The flow equation is purely diffusive (elliptic), while the ADE includes advection (hyperbolic character) — this makes it prone to oscillations and numerical dispersion if the grid or time step is too coarse.

Two dimensionless numbers control stability:

### Grid Peclet number

$$Pe_{grid} = \frac{v \cdot \Delta x}{D_L} \leq 2$$

This is a **cell-scale** requirement. It says: the advective transport across one cell must not exceed twice the dispersive spreading. Rearranging for maximum cell size:

$$\Delta x \leq \frac{2 \cdot D_L}{v} = 2 \cdot \alpha_L$$

since $D_L = \alpha_L \cdot v$ when molecular diffusion is negligible.

### Courant number

$$Cr = \frac{v \cdot \Delta t}{\Delta x} \leq 1$$

This is a **time-step** requirement. It says: a parcel of solute should not cross more than one cell per time step.

### Worked example with Limmat Valley values

Using domain-average values from NB2: $v \approx 2.3$ m/d, $\alpha_L = 10$ m, so $D_L = \alpha_L \cdot v \approx 23$ m$^2$/d.

| Criterion | Formula | Result |
|---|---|---|
| Max cell size | $\Delta x \leq 2 \alpha_L$ | $\leq 20$ m |
| Max time step (for $\Delta x = 20$ m) | $\Delta t \leq \Delta x / v$ | $\leq 8.7$ days |

Near the doublet wells, local velocities can reach ~7 m/d (the flow converges toward the abstraction well), reducing the Courant limit to $\leq 2.9$ days for the same cell size. With monthly stress periods ($\Delta t = 30$ days) and typical cell sizes of 25–50 m, Courant numbers can exceed 1 in the fast-flow zone — this motivates using the **TVD scheme** (see below).

Compare this to the flow model, where cell sizes of 50–100 m and time steps of days to weeks are common. **Transport demands finer spatial and temporal resolution.**

> ℹ️ **A note on units.** Molecular diffusion in water is $D_m^* \approx 1\times10^{-9}$ m²/s. Because the model runs in **days**, convert before use: $D_m^* \approx 8.64\times10^{-5}$ m²/d (the m/s → m/d factor is ×86 400). At $\alpha_L = 10$ m the mechanical term $\alpha_L v$ dominates, so diffusion is negligible here — but the unit conversion still matters when you enter `diffc` in the DSP package.

### TVD schemes and relaxation

The **TVD (Total Variation Diminishing)** advection scheme available in MODFLOW 6 is less sensitive to the grid Peclet constraint. It can handle $Pe_{grid} > 2$ without spurious oscillations, at the cost of introducing some numerical dispersion. In practice, TVD allows coarser grids than the classical $Pe \leq 2$ rule — but the Courant criterion still applies.

In [ ]:
# Checkpoint 2 — Peclet Calculation
check_task_with_solution('task_t03_checkpoint_2')

---

## 3 — MODFLOW 6 GWT Packages

MODFLOW 6 implements solute transport through the **Groundwater Transport (GWT)** model. Like GWF, GWT is built from interchangeable packages — each handles one physical process or boundary type.

| Package | Purpose | Doublet-study use |
|---|---|---|
| **MST** | Mass storage and transfer: effective porosity $n_e$ (+ optional sorption $K_d$, 1st-order decay) | $n_e = 0.20$; conservative tracer ⇒ no sorption/decay |
| **ADV** | Advection scheme selection (upstream, central, TVD) | `scheme='TVD'` for stability on the doublet grid |
| **DSP** | Dispersion: $\alpha_L$ (`alh`), $\alpha_T$ (`ath1`), molecular diffusion $D_m^*$ (`diffc`) | $\alpha_L = 10$ m, $\alpha_T = 1$ m, $D_m^* = 8.64\times10^{-5}$ m²/d |
| **SSM** | Source-sink mixing: concentration carried by each GWF flux | RIV / RCH inflow concentrations; injection-well concentration via AUX on WEL |
| **CNC** | Constant concentration (Dirichlet BC) | Only if a fixed-concentration boundary is genuinely required |
| **SRC** | Mass source/loading: per-cell mass-loading **rate** [M/T] | A spill/leak that injects mass without an associated water flux |
| **IC** | Initial concentration distribution | Background ≈ 0 (clean aquifer) |
| **OC** | Output control (what to save, when) | Concentration fields, mass budgets |

### How GWT couples to GWF

The GWT model receives the velocity field from GWF through either:
- A **GWF-GWT Exchange** (both models in the same simulation), or
- The **FMI** (Flow Model Interface) package, which reads a saved GWF flow solution from file

In both cases, **GWT never solves the flow equation** — it only solves the solute-transport equation using velocities computed by GWF.

### Specifying the source: WEL+SSM vs. CNC vs. SRC

How you introduce the tracer matters, and the three options are **not** interchangeable:

| Approach | What you control | What the model does | When to use |
|---|---|---|---|
| **WEL + SSM (AUX concentration)** | Injected concentration $c_{inj}$ on a pumping well | Mass rate = $Q_{inj}\cdot c_{inj}$ — **controlled, grid-independent** | The doublet injection well — **preferred** |
| **CNC** (constant concentration) | A fixed concentration in the cell | **Mass flux floats**: the model supplies whatever mass holds $C$ fixed → **grid- and flow-dependent, effectively unbounded** | Rarely; only a true Dirichlet boundary |
| **SRC** (mass loading) | A mass-loading **rate** [M/T] directly | Adds that mass to the cell, independent of any water flux | A source with no associated injection flux |

> ⚠️ **The CNC pitfall.** CNC fixes the *concentration* and lets the *mass flux* adjust to whatever is needed — so the mass entering the aquifer depends on the grid and the local flow, and can be far larger than intended. For a well-defined injection, use **WEL + SSM**, where the mass rate is exactly $Q_{inj}\cdot c_{inj}$. (This is *not* a "cell-area scaling" issue — that is a separate concern for area-based SRC/recharge loadings.)

### Common misconceptions

| Misconception | Reality |
|---|---|
| "Transport needs hydraulic conductivity $K$" | No — GWT receives the velocity field from GWF. It never sees $K$. |
| "Porosity is inherited from GWF" | No — effective porosity must be set in **MST**. GWF may use a different (total) porosity for storage; transport needs $n_e$. |
| "An injection well sets a fixed concentration" | No — WEL+SSM sets a **mass rate** $Q_{inj}\cdot c_{inj}$. Fixing concentration is CNC, with the floating-mass pitfall above. |
| "Finer time steps are always better" | The Courant criterion sets a *maximum* $\Delta t$. Below that limit, smaller steps waste computation without improving accuracy. |
| "RIV captures all river–aquifer solute exchange" | No — SSM only carries solute on the **advective** water flux. There is no diffusive exchange across the riverbed at zero net flow. |

<details>
<summary><strong>Optional: GWE (heat) background — the same machinery for temperature</strong> (click to expand)</summary>

The doublet wells in this study are physically a **groundwater heat-exchange (GWHE)** installation: paired injection–abstraction wells that move thermal energy for heating and cooling. Here we study the **conservative solute** that travels with the water, but the *same* MODFLOW 6 transport machinery can model the **heat** directly through the **Groundwater Energy (GWE)** model. You will not build a GWE model in this track — this box is background so you recognise it.

### Heat transport form of the ADE

For heat, concentration $C$ becomes temperature $T$, and energy is stored in **both** the water and the solid grains:

$$\underbrace{\frac{\partial}{\partial t}\bigl[n \rho_w c_w T + (1-n) \rho_s c_s T\bigr]}_{\text{bulk thermal storage}} = \underbrace{\nabla \cdot (\lambda_{bulk} \nabla T)}_{\text{conduction}} + \underbrace{\nabla \cdot (n \rho_w c_w \mathbf{D}_{mech} \nabla T)}_{\text{mechanical dispersion}} - \underbrace{\nabla \cdot (n \rho_w c_w \mathbf{v} T)}_{\text{advection}} + q_E$$

Because heat is stored in both phases, it experiences **thermal retardation** ($R \approx 3.2$ for the Limmat Valley gravels). A conservative solute is stored only in the water, so it is **not** retarded — the clean, simplest case, which is why we use it as the core path.

### GWE ↔ GWT package map

The two model types are structurally almost identical; only the "diffusive" term and the storage term differ:

| GWT package (solute) | GWE package (heat) | What changes |
|---|---|---|
| **MST** (Mass Storage and Transfer) | **EST** (Energy Storage and Transfer) | Sorption $K_d$, decay → heat capacity & density (thermal retardation) |
| **DSP** (Dispersion) | **CND** (Conduction-Dispersion) | Molecular diffusion $D_m^*$ → thermal conductivity $\lambda_{bulk}$ |
| **CNC** (Constant Concentration) | **CTP** (Constant Temperature) | Concentration value → temperature value |
| **SRC** (Mass Source/Loading) | **ESL** (Energy Source/Loading) | Mass rate [M/T] → energy rate [E/T] |
| **ADV** (Advection) | **ADV** | Identical — same scheme, same velocity field |
| **SSM** (Source-Sink Mixing) | **SSM** | Identical — associates concentration / temperature with each GWF stress |
| **IC** (Initial Conditions) | **IC** | Concentration → temperature |
| **OC** (Output Control) | **OC** | Identical |

**Key insight:** mechanical dispersion ($\alpha_L$, $\alpha_T$) is identical in GWT and GWE — it describes velocity-driven spreading that does not depend on *what* is transported. Only the molecular term differs (diffusion $D_m^*$ for solutes, conduction $\lambda_{bulk}$ for heat), and heat adds dual-phase storage (retardation). This is why dispersivity learned from one transfers to the other.

</details>

---

## 4 — Translating the Perceptual Model to GWT Packages

In NB2, we identified the key transport features of the doublet study. Here is how each maps to a MODFLOW 6 GWT package:

| Perceptual feature (from NB2) | Value | GWT package | How it enters the model |
|---|---|---|---|
| Effective porosity $n_e$ | 0.20 | **MST** | `porosity` input; controls pore velocity (and storage) — must equal $n_e$ |
| Longitudinal dispersivity $\alpha_L$ | 10 m | **DSP** | `alh` — longitudinal mechanical dispersion |
| Transverse dispersivity $\alpha_T$ | 1 m | **DSP** | `ath1` — transverse dispersion (single layer) |
| Molecular diffusion $D_m^*$ | $8.64\times10^{-5}$ m²/d | **DSP** | `diffc` — negligible here, but specified for completeness |
| Background concentration | ≈ 0 | **IC** | Initial concentration field (clean aquifer) |
| Injection-well concentration $c_{inj}$ | source strength | **SSM** (AUX on WEL) | Mass rate $= Q_{inj}\cdot c_{inj}$ — controlled, grid-independent |
| River / recharge inflow concentration | scenario-dependent | **SSM** | Auxiliary concentration on RIV / RCH packages |
| Advection scheme | TVD | **ADV** | `scheme='TVD'` for stability |
| (Optional) sorption $K_d$ / decay | off (conservative) | **MST** | Only for the optional reactive extension |

> 🤔 **Reflection:** The operational question for the doublet is **recirculation** — does the tracer injected at one well short-circuit back to the paired abstraction well, and how fast? In the model, the **SSM** package handles the source automatically: it assigns the injected concentration $c_{inj}$ to every unit of water the WEL package puts into the aquifer, so the mass rate is exactly $Q_{inj}\cdot c_{inj}$. No separate "concentration boundary" is needed, and — unlike CNC — the injected mass stays controlled and grid-independent.

In [ ]:
# Checkpoint 3 — Package Identification
create_multiple_choice('task_t03_checkpoint_3')

---

## Summary

In this notebook, you learned:

- The **advection-dispersion equation** extends the flow equation to transport: advection moves the front, dispersion spreads it, and sources/sinks add or remove mass — a **conservative tracer** is the clean core case (no sorption, no decay)
- The **1D ADE** has the analytical **Ogata–Banks** solution (full two-term form), but the convergent doublet flow is **not** 1D-uniform — so a clean match is reserved for a 1D verification toy in Step 8, and the doublet's pore-volume time is a **mean residence time**, not first arrival
- **Numerical stability** requires grid Peclet $\leq 2$ (max cell size $= 2\alpha_L$) and Courant $\leq 1$ (max time step $= \Delta x / v$) — transport needs finer grids and shorter time steps than flow; TVD relaxes the Peclet constraint
- MODFLOW 6 **GWT packages** (MST, ADV, DSP, SSM, CNC, SRC, IC, OC) each handle one aspect of solute transport; the source is best set with **WEL + SSM** ($Q_{inj}\cdot c_{inj}$), not CNC
- The perceptual model from NB2 **maps directly** to GWT inputs: $n_e \rightarrow$ MST; $\alpha_L$, $\alpha_T \rightarrow$ DSP (`alh`, `ath1`); injection source $\rightarrow$ SSM (AUX on WEL); background $\rightarrow$ IC

### What You're Taking Forward

| Concept | You'll use it in… |
|---|---|
| ADE / Peclet / Courant criteria | NB4: grid and time-step diagnosis for the transport model |
| GWT packages (MST, DSP, ADV, SSM) | NB4: FloPy implementation of each package |
| Perceptual → GWT package mapping | NB4: parameter assignment from NB2 values |
| WEL+SSM source vs. CNC pitfall | NB4: setting the injection-well concentration correctly |

### Documentation Resources

Bookmark these — you'll need them in NB4:

- **MODFLOW 6 GWT documentation:** [USGS MODFLOW 6 Description of Input and Output](https://modflow6.readthedocs.io/en/latest/)
- **FloPy documentation:** [FloPy — Python interface to MODFLOW](https://flopy.readthedocs.io/en/latest/)

---

## Next Steps

You now have the conceptual framework for solute-transport modelling in MODFLOW 6. In the next step, **04t builds the GWT model** — you implement each package in FloPy on the inherited flow grid.

**Continue to:** [Step 4 — Model Implementation](./04t_model_implementation.ipynb) — Build the GWT model in FloPy

**Review if needed:** [Step 2 — Perceptual Model](./02t_perceptual_model.ipynb) — Transport parameters and source concentrations